# Notebook 03 -- Category Harmonization

**Thesis:** Content Moderation on TikTok and X -- A Comparative Analysis of DSA Statements of Reasons (2025)  
**Author:** Mohsen Zahedi  

> **Note:** This notebook is pre-run. Data lives at `C:\BA_Data\` and is not included in this repository.

---

This notebook documents the category harmonization step. The core problem is that the DSA API changed its `category` vocabulary on July 1, 2025 (v1 to v2), making direct comparison of records across that boundary impossible without a mapping layer.

In [9]:
from pathlib import Path

import polars as pl

DATA_ROOT = Path(r"C:\BA_Data")
TIKTOK_RAW = DATA_ROOT / "tiktok_de_2025.parquet"
X_RAW = DATA_ROOT / "x_de_2025.parquet"
TIKTOK_HARMONIZED = DATA_ROOT / "tiktok_de_2025_harmonized.parquet"
X_HARMONIZED = DATA_ROOT / "x_de_2025_harmonized.parquet"

# Mapping table is part of the repository
REPO_ROOT = Path("__file__").resolve().parents[1] if "__file__" in dir() else Path(".").resolve().parent
MAPPING_CSV = REPO_ROOT / "03_harmonization" / "taxonomy_mapping_v1_v2.csv"

## 1. The v1/v2 Schema Break

On **July 1, 2025** the DSA Commission released a new submission API (v2) that modified the `category` field:

- Some categories were **renamed** (e.g. `SCOPE_OF_PLATFORM_SERVICE` became `OTHER_VIOLATION_TC`)
- One category was **removed** (`NON_CONSENSUAL_BEHAVIOUR`, no v2 equivalent)
- One category was officially **introduced** in v2 (`CYBER_VIOLENCE`), though it already appeared in some v1 records

[DSA Official change logs](https://transparency.dsa.ec.europa.eu/page/api-documentation#changelog)

Without harmonization, any category-level comparison between the v1 period (Jan--Jun 2025) and the v2 period (Jul--Dec 2025) would be misleading.

In [10]:
# Raw category labels in TikTok, split by api_version
raw_cats = (
    pl.scan_parquet(str(TIKTOK_RAW))
    .group_by(["api_version", "category"])
    .agg(pl.len().alias("count"))
    .sort(["api_version", "count"], descending=[False, True])
    .collect(engine="streaming")
)

print("TikTok raw categories per api_version:")
with pl.Config(fmt_str_lengths=100, tbl_rows=100):
    display(raw_cats)

TikTok raw categories per api_version:


api_version,category,count
str,str,u32
"""v1""","""STATEMENT_CATEGORY_ILLEGAL_OR_HARMFUL_SPEECH""",399152169
"""v1""","""STATEMENT_CATEGORY_SCOPE_OF_PLATFORM_SERVICE""",328623461
"""v1""","""STATEMENT_CATEGORY_NEGATIVE_EFFECTS_ON_CIVIC_DISCOURSE_OR_ELECTIONS""",37292141
"""v1""","""STATEMENT_CATEGORY_VIOLENCE""",29874258
"""v1""","""STATEMENT_CATEGORY_PROTECTION_OF_MINORS""",12604612
"""v1""","""STATEMENT_CATEGORY_SCAMS_AND_FRAUD""",11372842
"""v1""","""STATEMENT_CATEGORY_PORNOGRAPHY_OR_SEXUALIZED_CONTENT""",5714374
"""v1""","""STATEMENT_CATEGORY_DATA_PROTECTION_AND_PRIVACY_VIOLATIONS""",3616694
"""v1""","""STATEMENT_CATEGORY_SELF_HARM""",973242


In [11]:
# Raw category labels in X, split by api_version
raw_cats_x = (
    pl.scan_parquet(str(X_RAW))
    .group_by(["api_version", "category"])
    .agg(pl.len().alias("count"))
    .sort(["api_version", "count"], descending=[False, True])
    .collect()
)

print("X raw categories per api_version:")
with pl.Config(fmt_str_lengths=100, tbl_rows=100):
    display(raw_cats_x)

X raw categories per api_version:


api_version,category,count
str,str,u32
"""v1""","""STATEMENT_CATEGORY_SCOPE_OF_PLATFORM_SERVICE""",60974
"""v1""","""STATEMENT_CATEGORY_PORNOGRAPHY_OR_SEXUALIZED_CONTENT""",32189
"""v1""","""STATEMENT_CATEGORY_PROTECTION_OF_MINORS""",13643
"""v1""","""STATEMENT_CATEGORY_SCAMS_AND_FRAUD""",8290
"""v1""","""STATEMENT_CATEGORY_VIOLENCE""",6357
"""v1""","""STATEMENT_CATEGORY_UNSAFE_AND_ILLEGAL_PRODUCTS""",5146
"""v1""","""STATEMENT_CATEGORY_NON_CONSENSUAL_BEHAVIOUR""",2103
"""v1""","""STATEMENT_CATEGORY_SELF_HARM""",2084
"""v1""","""STATEMENT_CATEGORY_INTELLECTUAL_PROPERTY_INFRINGEMENTS""",906


## 1b. How Many Unique Categories Exist Per Platform?

Before harmonization, both platforms report violations using DSA-defined category labels. The label set changed between v1 (before July 1, 2025) and v2 (from July 1, 2025 onwards). The cell below counts how many distinct category labels each platform actually used in each API version.

In [12]:
for label, df in [("TikTok", raw_cats), ("X", raw_cats_x)]:
    print(f"{label}:")
    summary = (
        df.group_by("api_version")
        .agg(pl.col("category").n_unique().alias("unique_categories"))
        .sort("api_version")
    )
    for row in summary.iter_rows(named=True):
        print(f"  {row['api_version']}: {row['unique_categories']} unique category labels")
    print()

TikTok:
  v1: 14 unique category labels
  v2: 15 unique category labels

X:
  v1: 16 unique category labels
  v2: 12 unique category labels



## 1c. What Changed Between v1 and v2?

Categories that appear **only in v1** were either removed or renamed in v2. Categories **only in v2** are either new or renamed from v1. Categories **in both** were carried over unchanged. This comparison -- done per platform -- shows exactly why a mapping layer is needed before any cross-period analysis.

In [13]:
for label, df in [("TikTok", raw_cats), ("X", raw_cats_x)]:
    v1 = set(df.filter(pl.col("api_version") == "v1")["category"].to_list())
    v2 = set(df.filter(pl.col("api_version") == "v2")["category"].to_list())

    only_v1 = sorted(v1 - v2)
    only_v2 = sorted(v2 - v1)
    in_both = sorted(v1 & v2)

    print(f"{'=' * 65}")
    print(f"  {label}")
    print(f"{'=' * 65}")
    print(f"\n  Only in v1 -- removed or renamed in v2 ({len(only_v1)}):")
    for c in only_v1:
        print(f"    - {c}")
    print(f"\n  Only in v2 -- new or renamed from v1 ({len(only_v2)}):")
    for c in only_v2:
        print(f"    - {c}")
    print(f"\n  In both v1 and v2 -- unchanged ({len(in_both)}):")
    for c in in_both:
        print(f"    - {c}")
    print()

  TikTok

  Only in v1 -- removed or renamed in v2 (2):
    - STATEMENT_CATEGORY_NON_CONSENSUAL_BEHAVIOUR
    - STATEMENT_CATEGORY_UNSAFE_AND_ILLEGAL_PRODUCTS

  Only in v2 -- new or renamed from v1 (3):
    - STATEMENT_CATEGORY_CYBER_VIOLENCE
    - STATEMENT_CATEGORY_OTHER_VIOLATION_TC
    - STATEMENT_CATEGORY_UNSAFE_AND_PROHIBITED_PRODUCTS

  In both v1 and v2 -- unchanged (12):
    - STATEMENT_CATEGORY_ANIMAL_WELFARE
    - STATEMENT_CATEGORY_DATA_PROTECTION_AND_PRIVACY_VIOLATIONS
    - STATEMENT_CATEGORY_ILLEGAL_OR_HARMFUL_SPEECH
    - STATEMENT_CATEGORY_INTELLECTUAL_PROPERTY_INFRINGEMENTS
    - STATEMENT_CATEGORY_NEGATIVE_EFFECTS_ON_CIVIC_DISCOURSE_OR_ELECTIONS
    - STATEMENT_CATEGORY_PORNOGRAPHY_OR_SEXUALIZED_CONTENT
    - STATEMENT_CATEGORY_PROTECTION_OF_MINORS
    - STATEMENT_CATEGORY_RISK_FOR_PUBLIC_SECURITY
    - STATEMENT_CATEGORY_SCAMS_AND_FRAUD
    - STATEMENT_CATEGORY_SCOPE_OF_PLATFORM_SERVICE
    - STATEMENT_CATEGORY_SELF_HARM
    - STATEMENT_CATEGORY_VIOLENCE

  X

  On

## 2. Taxonomy Mapping Table

A mapping table (`03_harmonization/taxonomy_mapping_v1_v2.csv`) was created to assign each raw category label to a **Super-Cluster** -- a stable, version-independent label that works across both v1 and v2 records.

The table was built by:
1. Extracting all unique category labels per api_version from TikTok and X
2. Cross-referencing the official DSA API changelog (July 2025)
3. Assigning each label to a Super-Cluster based on semantic proximity

**Script:** `03_harmonization/scripts/harmonize_categories_polars.py`

In [14]:
mapping = pl.read_csv(str(MAPPING_CSV))

with pl.Config(fmt_str_lengths=100, tbl_rows=100):
    display(mapping)

original_category,api_version,super_cluster,mapping_type,dsa_legal_basis,notes
str,str,str,str,str,str
"""STATEMENT_CATEGORY_ANIMAL_WELFARE""","""both""","""ANIMAL_WELFARE""","""direct""","""DSA Art. 3(h)""","""Identical in v1 and v2"""
"""STATEMENT_CATEGORY_CYBER_VIOLENCE""","""both""","""CYBER_VIOLENCE""","""present_in_both""","""DSA Changelog July 2025""","""Officially new in v2 but appears in v1 records - document as anomaly"""
"""STATEMENT_CATEGORY_DATA_PROTECTION_AND_PRIVACY_VIOLATIONS""","""both""","""PRIVACY_AND_DATA""","""direct""","""DSA Art. 3(h)""","""Identical in v1 and v2"""
"""STATEMENT_CATEGORY_ILLEGAL_OR_HARMFUL_SPEECH""","""both""","""HARMFUL_SPEECH""","""direct""","""DSA Art. 3(h)""","""Identical in v1 and v2"""
"""STATEMENT_CATEGORY_INTELLECTUAL_PROPERTY_INFRINGEMENTS""","""both""","""INTELLECTUAL_PROPERTY""","""direct""","""DSA Art. 3(h)""","""Identical in v1 and v2"""
"""STATEMENT_CATEGORY_NEGATIVE_EFFECTS_ON_CIVIC_DISCOURSE_OR_ELECTIONS""","""both""","""CIVIC_AND_ELECTIONS""","""direct""","""DSA Art. 3(h)""","""Identical in v1 and v2"""
"""STATEMENT_CATEGORY_OTHER_VIOLATION_TC""","""both""","""OTHER""","""renamed_from_v1""","""DSA Changelog July 2025""","""Maps from v1 SCOPE_OF_PLATFORM_SERVICE"""
"""STATEMENT_CATEGORY_PORNOGRAPHY_OR_SEXUALIZED_CONTENT""","""both""","""SEXUAL_CONTENT""","""direct""","""DSA Art. 3(h)""","""Appears in both despite being removed in official v2 schema"""
"""STATEMENT_CATEGORY_PROTECTION_OF_MINORS""","""both""","""PROTECTION_OF_MINORS""","""direct""","""DSA Art. 3(h)""","""Identical in v1 and v2"""


## 3. Super-Cluster Taxonomy

13 Super-Clusters were defined. Two mapping decisions required explicit justification:

| Decision | Rationale |
|---|---|
| `RISK_FOR_PUBLIC_SECURITY` grouped into **VIOLENCE** | Both involve physical harm. Semantic proximity justifies merging. |
| `CYBER_VIOLENCE` kept as **separate Super-Cluster** | Represents psychological/reputational harm, distinct from physical violence. Also analytically important as a v2-introduced category with a dramatic rise in X data. |
| `NON_CONSENSUAL_BEHAVIOUR` (v1 only) grouped into **SEXUAL_CONTENT** | No v2 equivalent exists. Closest semantic match is SEXUAL_CONTENT. |

All 13 Super-Clusters:

In [15]:
super_clusters = mapping.select("super_cluster").unique().sort("super_cluster")
for sc in super_clusters.to_series().to_list():
    print(f"  - {sc}")

  - ANIMAL_WELFARE
  - CIVIC_AND_ELECTIONS
  - CYBER_VIOLENCE
  - HARMFUL_SPEECH
  - INTELLECTUAL_PROPERTY
  - OTHER
  - PRIVACY_AND_DATA
  - PROTECTION_OF_MINORS
  - SCAMS_AND_FRAUD
  - SELF_HARM
  - SEXUAL_CONTENT
  - UNSAFE_PRODUCTS
  - VIOLENCE


## 3b. Full Super-Cluster Mapping

The table below shows every original DSA category, the Super-Cluster it belongs to, and the DSA legal basis for the grouping. The `mapping_type` column explains the relationship:

| mapping_type | Meaning |
|---|---|
| `direct` | Label is identical in v1 and v2 |
| `renamed_in_v2` | v1 label was renamed in v2 -- both map to the same Super-Cluster |
| `renamed_from_v1` | v2 label that replaced a v1 label |
| `removed_in_v2` | v1 label with no v2 equivalent -- mapped by semantic proximity |
| `present_in_both` | Category appears in both versions despite being officially scoped to one -- treated as an anomaly (see Section 7) |

In [16]:
sc_breakdown = (
    mapping
    .select(["super_cluster", "original_category", "api_version", "mapping_type", "dsa_legal_basis"])
    .sort(["super_cluster", "api_version", "original_category"])
)
with pl.Config(fmt_str_lengths=120, tbl_rows=50):
    display(sc_breakdown)

super_cluster,original_category,api_version,mapping_type,dsa_legal_basis
str,str,str,str,str
"""ANIMAL_WELFARE""","""STATEMENT_CATEGORY_ANIMAL_WELFARE""","""both""","""direct""","""DSA Art. 3(h)"""
"""CIVIC_AND_ELECTIONS""","""STATEMENT_CATEGORY_NEGATIVE_EFFECTS_ON_CIVIC_DISCOURSE_OR_ELECTIONS""","""both""","""direct""","""DSA Art. 3(h)"""
"""CYBER_VIOLENCE""","""STATEMENT_CATEGORY_CYBER_VIOLENCE""","""both""","""present_in_both""","""DSA Changelog July 2025"""
"""HARMFUL_SPEECH""","""STATEMENT_CATEGORY_ILLEGAL_OR_HARMFUL_SPEECH""","""both""","""direct""","""DSA Art. 3(h)"""
"""INTELLECTUAL_PROPERTY""","""STATEMENT_CATEGORY_INTELLECTUAL_PROPERTY_INFRINGEMENTS""","""both""","""direct""","""DSA Art. 3(h)"""
"""OTHER""","""STATEMENT_CATEGORY_OTHER_VIOLATION_TC""","""both""","""renamed_from_v1""","""DSA Changelog July 2025"""
"""OTHER""","""STATEMENT_CATEGORY_SCOPE_OF_PLATFORM_SERVICE""","""both""","""renamed_to_v2""","""DSA Changelog July 2025"""
"""PRIVACY_AND_DATA""","""STATEMENT_CATEGORY_DATA_PROTECTION_AND_PRIVACY_VIOLATIONS""","""both""","""direct""","""DSA Art. 3(h)"""
"""PROTECTION_OF_MINORS""","""STATEMENT_CATEGORY_PROTECTION_OF_MINORS""","""both""","""direct""","""DSA Art. 3(h)"""


## 3c. Which Original Categories Map to Each Super-Cluster -- Per Platform?

Not all DSA categories appear in both platforms. The table below joins each platform's actual raw category records with the mapping table to show how many records fall under each Super-Cluster per API version. This reveals which Super-Clusters are analytically meaningful for each platform.

In [17]:
for label, df in [("TikTok", raw_cats), ("X", raw_cats_x)]:
    breakdown = (
        df
        .join(
            mapping.select(["original_category", "super_cluster"]),
            left_on="category",
            right_on="original_category",
            how="left"
        )
        .group_by(["super_cluster", "api_version"])
        .agg(pl.col("count").sum().alias("total_records"))
        .sort(["super_cluster", "api_version"])
    )
    print(f"\n{label} -- records per Super-Cluster per api_version:")
    with pl.Config(fmt_str_lengths=60, tbl_rows=50):
        display(breakdown)


TikTok -- records per Super-Cluster per api_version:


super_cluster,api_version,total_records
str,str,u32
"""ANIMAL_WELFARE""","""v1""",948719
"""ANIMAL_WELFARE""","""v2""",198251
"""CIVIC_AND_ELECTIONS""","""v1""",37292141
"""CIVIC_AND_ELECTIONS""","""v2""",8247264
"""CYBER_VIOLENCE""","""v2""",1389
"""HARMFUL_SPEECH""","""v1""",399152169
"""HARMFUL_SPEECH""","""v2""",35245452
"""INTELLECTUAL_PROPERTY""","""v1""",45491
"""INTELLECTUAL_PROPERTY""","""v2""",94441



X -- records per Super-Cluster per api_version:


super_cluster,api_version,total_records
str,str,u32
"""ANIMAL_WELFARE""","""v1""",10
"""ANIMAL_WELFARE""","""v2""",3
"""CYBER_VIOLENCE""","""v1""",34
"""CYBER_VIOLENCE""","""v2""",19143
"""HARMFUL_SPEECH""","""v1""",167
"""HARMFUL_SPEECH""","""v2""",125
"""INTELLECTUAL_PROPERTY""","""v1""",906
"""INTELLECTUAL_PROPERTY""","""v2""",1018
"""OTHER""","""v1""",60975


## 3d. Schema Anomalies

Two categories appeared outside their officially defined API version scope. These are real platform submissions using an unexpected label -- not data errors. Both are documented in the cleaning decisions log (`CD-20260429-02`) and preserved in `category_raw`. The `category_harmonized` column handles them correctly via the mapping table.

| Category | Expected version | Found in | Interpretation |
|---|---|---|---|
| `STATEMENT_CATEGORY_CYBER_VIOLENCE` | v2 only | v1 records | Platforms submitted this label before it was officially introduced |
| `STATEMENT_CATEGORY_PORNOGRAPHY_OR_SEXUALIZED_CONTENT` | v1 only | v2 records | Platforms continued using a removed label after the v2 transition |

In [18]:
anomalies = [
    ("STATEMENT_CATEGORY_CYBER_VIOLENCE", "v1", "officially a v2-only category"),
    ("STATEMENT_CATEGORY_PORNOGRAPHY_OR_SEXUALIZED_CONTENT", "v2", "officially removed in v2"),
]

for label, raw_df in [("TikTok", raw_cats), ("X", raw_cats_x)]:
    print(f"{label}:")
    for category, version, note in anomalies:
        match = raw_df.filter(
            (pl.col("api_version") == version) & (pl.col("category") == category)
        )
        count = match["count"].sum() if len(match) > 0 else 0
        print(f"  {category}")
        print(f"    found in {version} ({note}): {count:,} records")
    print()

TikTok:
  STATEMENT_CATEGORY_CYBER_VIOLENCE
    found in v1 (officially a v2-only category): 0 records
  STATEMENT_CATEGORY_PORNOGRAPHY_OR_SEXUALIZED_CONTENT
    found in v2 (officially removed in v2): 7 records

X:
  STATEMENT_CATEGORY_CYBER_VIOLENCE
    found in v1 (officially a v2-only category): 34 records
  STATEMENT_CATEGORY_PORNOGRAPHY_OR_SEXUALIZED_CONTENT
    found in v2 (officially removed in v2): 0 records



## 4. Before and After Harmonization

The harmonization script added two new columns to each platform's working file:

- `category_harmonized` -- the Super-Cluster label
- `application_date_harmonized` -- date normalized to UTC with the time component removed

The original columns (`category` and `application_date`) were preserved as `category_raw` and `application_date_raw` for audit purposes.

In [19]:
# Show a sample with raw and harmonized columns side by side
(
    pl.scan_parquet(str(TIKTOK_HARMONIZED))
    .select(["api_version", "category_raw", "category_harmonized", "application_date_raw", "application_date_harmonized"])
    .head(8)
    .collect()
)

api_version,category_raw,category_harmonized,application_date_raw,application_date_harmonized
str,str,str,date,date
"""v1""","""STATEMENT_CATEGORY_VIOLENCE""","""VIOLENCE""",2025-01-01,2025-01-01
"""v1""","""STATEMENT_CATEGORY_VIOLENCE""","""VIOLENCE""",2025-01-01,2025-01-01
"""v1""","""STATEMENT_CATEGORY_VIOLENCE""","""VIOLENCE""",2025-01-01,2025-01-01
"""v1""","""STATEMENT_CATEGORY_VIOLENCE""","""VIOLENCE""",2025-01-01,2025-01-01
"""v1""","""STATEMENT_CATEGORY_VIOLENCE""","""VIOLENCE""",2025-01-01,2025-01-01
"""v1""","""STATEMENT_CATEGORY_VIOLENCE""","""VIOLENCE""",2025-01-01,2025-01-01
"""v1""","""STATEMENT_CATEGORY_VIOLENCE""","""VIOLENCE""",2025-01-01,2025-01-01
"""v1""","""STATEMENT_CATEGORY_VIOLENCE""","""VIOLENCE""",2025-01-01,2025-01-01


In [20]:
# X -- raw and harmonized columns side by side
(
    pl.scan_parquet(str(X_HARMONIZED))
    .select(["api_version", "category_raw", "category_harmonized", "application_date_raw", "application_date_harmonized"])
    .head(8)
    .collect()
)

api_version,category_raw,category_harmonized,application_date_raw,application_date_harmonized
str,str,str,date,date
"""v2""","""STATEMENT_CATEGORY_UNSAFE_AND_…","""UNSAFE_PRODUCTS""",2025-07-06,2025-07-06
"""v2""","""STATEMENT_CATEGORY_UNSAFE_AND_…","""UNSAFE_PRODUCTS""",2025-07-06,2025-07-06
"""v2""","""STATEMENT_CATEGORY_UNSAFE_AND_…","""UNSAFE_PRODUCTS""",2025-07-06,2025-07-06
"""v2""","""STATEMENT_CATEGORY_UNSAFE_AND_…","""UNSAFE_PRODUCTS""",2025-07-06,2025-07-06
"""v2""","""STATEMENT_CATEGORY_UNSAFE_AND_…","""UNSAFE_PRODUCTS""",2025-07-06,2025-07-06
"""v2""","""STATEMENT_CATEGORY_SCAMS_AND_F…","""SCAMS_AND_FRAUD""",2025-07-06,2025-07-06
"""v2""","""STATEMENT_CATEGORY_UNSAFE_AND_…","""UNSAFE_PRODUCTS""",2025-07-06,2025-07-06
"""v2""","""STATEMENT_CATEGORY_UNSAFE_AND_…","""UNSAFE_PRODUCTS""",2025-07-06,2025-07-06


## 5. Category Distribution After Harmonization

With categories harmonized, distribution can be compared across v1 and v2 and across platforms.

In [21]:
# TikTok: category distribution per api_version
tiktok_dist = (
    pl.scan_parquet(str(TIKTOK_HARMONIZED))
    .group_by(["api_version", "category_harmonized"])
    .agg(pl.len().alias("count"))
    .sort(["api_version", "count"], descending=[False, True])
    .collect(engine="streaming")
)
print("TikTok -- harmonized category distribution:")
tiktok_dist

TikTok -- harmonized category distribution:


api_version,category_harmonized,count
str,str,u32
"""v1""","""HARMFUL_SPEECH""",399152169
"""v1""","""OTHER""",328623461
"""v1""","""CIVIC_AND_ELECTIONS""",37292141
"""v1""","""VIOLENCE""",29875606
"""v1""","""PROTECTION_OF_MINORS""",12604612
…,…,…
"""v2""","""ANIMAL_WELFARE""",198251
"""v2""","""INTELLECTUAL_PROPERTY""",94441
"""v2""","""CYBER_VIOLENCE""",1389


In [22]:
# X: category distribution per api_version
x_dist = (
    pl.scan_parquet(str(X_HARMONIZED))
    .group_by(["api_version", "category_harmonized"])
    .agg(pl.len().alias("count"))
    .sort(["api_version", "count"], descending=[False, True])
    .collect()
)
print("X -- harmonized category distribution:")
x_dist

X -- harmonized category distribution:


api_version,category_harmonized,count
str,str,u32
"""v1""","""OTHER""",60975
"""v1""","""SEXUAL_CONTENT""",34292
"""v1""","""PROTECTION_OF_MINORS""",13643
"""v1""","""SCAMS_AND_FRAUD""",8290
"""v1""","""VIOLENCE""",6361
…,…,…
"""v2""","""INTELLECTUAL_PROPERTY""",1018
"""v2""","""SELF_HARM""",1018
"""v2""","""PRIVACY_AND_DATA""",185


## 6. Key Findings from Harmonization

- **TikTok dominant category shifted dramatically:** `HARMFUL_SPEECH` dominated v1 (48.05%), but `OTHER` (mapping from `OTHER_VIOLATION_TC`) dominated v2 (67.27%). This reflects a genuine change in TikTok's reporting behavior after the schema transition, not a data artifact.
- **X CYBER_VIOLENCE exploded in v2:** 34 records in v1 vs 19,143 records in v2 (37.44% of all X v2 records). The clearest schema transition effect in the dataset.
- **Zero unmapped categories:** Validation confirmed every raw category label in both platforms was covered by the mapping table.